In [ ]:
# UltimateDocResearcher — test run success

In [ ]:
import subprocess, os

# Clone repo
repo = "Amitro123/UltimateDocResearcher"
if not os.path.exists("ultimate-doc-researcher"):
    subprocess.run(["git", "clone", f"https://github.com/{repo}.git", "ultimate-doc-researcher"], check=True)
os.chdir("ultimate-doc-researcher")

# Configure git for result commits
subprocess.run(["git", "config", "user.email", "kaggle@autoresearch.bot"], check=True)
subprocess.run(["git", "config", "user.name", "KaggleBot"], check=True)
print("✅ Repo cloned")


In [ ]:
# Install dependencies
subprocess.run([
    "pip", "install", "-q",
    "unsloth", "trl", "peft", "pymupdf",
    "aiohttp", "beautifulsoup4", "google-api-python-client", "google-auth",
], check=True)
print("✅ Dependencies installed")


In [ ]:
import sys
sys.path.insert(0, ".")
from collector.ultimate_collector import UltimateCollector
from autoresearch.prepare import prepare
from autoresearch.train import TrainConfig, research_loop

TOPIC = "test run success"
N_ITERATIONS = 1

# ── Step 1: Collect ──────────────────────────────────────────────────────
collector = UltimateCollector(
    google_queries=[
        f"{TOPIC} research paper",
        f"{TOPIC} best practices",
        f"{TOPIC} tutorial",
    ],
    reddit_subreddits=["MachineLearning", "LocalLLaMA", "artificial"],
    github_repos=["karpathy/autoresearch"],
    output_dir="data/",
)
docs = collector.run()
print(f"Collected {len(docs)} documents")


In [ ]:
# ── Step 2: Analyze / clean ─────────────────────────────────────────────
from collector.analyzer import analyze_corpus
report = analyze_corpus("data/all_docs.txt", "data/")
print(report)


In [ ]:
# ── Step 3: Prepare Q&A pairs ────────────────────────────────────────────
prepare(
    corpus_path="data/all_docs_cleaned.txt",
    output_dir="data/",
    program_md="templates/program.md",
    max_pairs=500,
    use_llm=False,  # set True if OPENAI_API_KEY is available
)


In [ ]:
# ── Step 4: Research loop ─────────────────────────────────────────────────
cfg = TrainConfig(
    model_name="unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
    num_train_epochs=1,
    topic=TOPIC,
    output_dir="/kaggle/working/models/lora",
    results_tsv="/kaggle/working/results.tsv",
)

for i in range(N_ITERATIONS):
    from autoresearch.train import train
    cfg.iteration = i + 1
    metrics = train(cfg)
    print(f"Iteration {i+1}: val_score={metrics.get('val_score', 'N/A')}")


In [ ]:
# ── Step 5: Show results ──────────────────────────────────────────────────
import pandas as pd
df = pd.read_csv("/kaggle/working/results.tsv", sep="\t")
print(df[["iteration", "val_loss", "val_score", "elapsed_seconds"]])
df.to_csv("/kaggle/working/results.tsv", sep="\t", index=False)
print("✅ Done — results.tsv saved as output")
